<a href="https://colab.research.google.com/github/rikesh28/Credit_Card_Fraud_Detection/blob/main/notebooks/6)%20Production_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score, precision_score, recall_score, f1_score, precision_recall_curve
import pickle

In [ ]:
train_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Fraud_Detection_System_Project/2) Data/Processed Data/train_df.csv')
test_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Fraud_Detection_System_Project/2) Data/Processed Data/test_df.csv')

In [ ]:
# Only features available at transaction time (no V/C/D features that need historical lookups)
api_features = [
    # raw transaction info
    'TransactionAmt', 'ProductCD',
    'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2',
    'P_emaildomain', 'R_emaildomain',
    # engineered features we can compute on the fly
    'TransactionAmt_log',
    'TransactionAmt_decimal',
    'is_round_amount',
    'email_domain_match',
    'P_email_is_common',
    'R_email_is_common',
    'has_P_email',
    'has_R_email',
]

print(f'{len(api_features)} features for API model')

In [ ]:
X_train = train_df[api_features].copy()
y_train = train_df['isFraud']

X_test = test_df[api_features].copy()
y_test = test_df['isFraud']

In [ ]:
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Fraud rate (train): {y_train.mean()*100:.2f}%")

X_train shape: (472432, 20)
X_test shape: (118108, 20)
Fraud rate (train): 3.51%


In [ ]:
scale_pos_W = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale_pos_W:.2f}")

Scale pos weight: 27.46


In [ ]:
api_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_W,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc'
)
api_model.fit(X_train, y_train, verbose=False)
print('Done')

In [ ]:
y_pred_proba = api_model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

print(classification_report(y_test, y_pred))
print(f'ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall: {recall_score(y_test, y_pred):.4f}')

# 0.82 AUC vs 0.90 for the full model. Lost ~8% AUC but this model
# can run in <100ms with no database lookups.

In [ ]:
# Check legitimate predictions
legit_mask = y_test == 0
legit_probs = y_pred_proba[legit_mask]

print(f"\n=== LEGITIMATE TRANSACTION CHECK ===")
print(f"Mean fraud probability: {legit_probs.mean():.4f}")
print(f"% with >60% fraud prob: {(legit_probs > 0.6).sum() / len(legit_probs) * 100:.2f}%")


=== LEGITIMATE TRANSACTION CHECK ===
Mean fraud probability: 0.2993
% with >60% fraud prob: 9.55%


In [ ]:
# Saving API Model
with open('/content/drive/MyDrive/Colab_Notebooks/Fraud_Detection_System_Project/3) Models/api_model_xgb.pkl','wb') as f:
  pickle.dump(api_model, f)

# Saving Features names
pd.DataFrame({'feature': api_features}).to_csv('/content/drive/MyDrive/Colab_Notebooks/Fraud_Detection_System_Project/3) Models/api_feature_names.csv', index = False)

print("\nAPI model saved as: api_model_xgb.pkl")
print("Features saved as: api_feature_names.csv")


API model saved as: api_model_xgb.pkl
Features saved as: api_feature_names.csv


In [ ]:
#Optimal Threshold Analysis

# Calculate precision and recall at different thresholds
precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba)

# Calculate F1 scores for each threshold
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)

# Find optimal threshold (max F1)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]
optimal_f1 = f1_scores[optimal_idx]

print(f"Default Threshold: 0.50")
print(f"Optimal Threshold: {optimal_threshold:.3f}")
print(f"Optimal F1 Score: {optimal_f1:.4f}")

# Make predictions with optimal threshold
y_pred_optimal = (y_pred_proba >= optimal_threshold).astype(int)

print(f"\n=== PERFORMANCE AT DIFFERENT THRESHOLDS ===")
print(f"{'Threshold':<15} {'Precision':<15} {'Recall':<15} {'F1-Score':<15}")
print("-" * 60)

for thresh in [0.3, 0.4, 0.5, optimal_threshold, 0.6, 0.7]:
    y_pred_thresh = (y_pred_proba >= thresh).astype(int)
    prec = precision_score(y_test, y_pred_thresh)
    rec = recall_score(y_test, y_pred_thresh)
    f1 = f1_score(y_test, y_pred_thresh)
    marker = " ← Optimal" if abs(thresh - optimal_threshold) < 0.01 else ""
    print(f"{thresh:<15.3f} {prec:<15.4f} {rec:<15.4f} {f1:<15.4f}{marker}")

Default Threshold: 0.50
Optimal Threshold: 0.757
Optimal F1 Score: 0.2974

=== PERFORMANCE AT DIFFERENT THRESHOLDS ===
Threshold       Precision       Recall          F1-Score       
------------------------------------------------------------
0.300           0.0655          0.8617          0.1218         
0.400           0.0874          0.7712          0.1570         
0.500           0.1251          0.6732          0.2110         
0.757           0.2494          0.3684          0.2974          ← Optimal
0.600           0.1656          0.5315          0.2525         
0.700           0.2180          0.4213          0.2873         


## Production model tradeoffs

The research model (434 features, 0.90 AUC) can't be deployed because many features
need historical database lookups that add latency and infrastructure cost.

This production model uses only 20 features available at transaction time:
- 434 -> 20 features (95% reduction)
- 0.90 -> 0.82 ROC-AUC (~8% drop)
- 500ms -> <100ms inference

The 8% AUC drop is the cost of deployability. In practice, you could partially
recover this by adding a batch re-scoring job that runs the full model offline
and flags additional fraud after the fact.